# Prototype Learning for Market State Classification

This notebook demonstrates the prototype learning approach for trading, implementing the ProtoPNet architecture adapted for financial time series.

## Contents
1. Data Loading (Stock & Crypto)
2. Feature Engineering
3. Model Training
4. Prototype Analysis
5. Backtesting

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from model import ProtoPNet, MarketState
from data_loader import BybitDataLoader, StockDataLoader, FeatureEngineering, prepare_data_for_training
from train import ProtoPNetTrainer
from backtest import PrototypeBacktester, print_backtest_report

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Data Loading

### Option A: Cryptocurrency Data from Bybit

In [ ]:
# Load cryptocurrency data from Bybit
bybit_loader = BybitDataLoader()

# Fetch BTC/USDT hourly data
crypto_data = bybit_loader.get_klines_history(
    symbol='BTCUSDT',
    interval='1h',
    start_time='2023-01-01',
    end_time='2024-01-01'
)

print(f"Crypto data shape: {crypto_data.shape}")
crypto_data.head()

### Option B: Stock Market Data from Yahoo Finance

In [ ]:
# Load stock data (requires yfinance: pip install yfinance)
try:
    stock_loader = StockDataLoader()
    stock_data = stock_loader.get_data(
        symbols='SPY',
        start='2020-01-01',
        end='2024-01-01',
        interval='1d'
    )
    stock_data = stock_data.reset_index()
    stock_data.columns = ['timestamp', 'open', 'high', 'low', 'close', 'adj_close', 'volume']
    print(f"Stock data shape: {stock_data.shape}")
    display(stock_data.head())
except ImportError:
    print("yfinance not installed. Using crypto data only.")
    stock_data = None

In [ ]:
# Use crypto data for this example (or stock_data if available)
df = crypto_data.copy()
print(f"Using dataset with {len(df)} samples")

## 2. Feature Engineering

In [ ]:
# Initialize feature engineering
lookback = 60  # Use 60 periods of history
fe = FeatureEngineering(lookback=lookback)

# Compute technical features
df = fe.compute_features(df)

# Label market states
df = fe.label_market_states(
    df,
    forward_returns_period=5,
    threshold_trend=0.02,
    threshold_breakout=0.03
)

# Show feature statistics
print("Computed features:")
print(df.columns.tolist())

In [ ]:
# Check label distribution
label_counts = df['label'].value_counts().sort_index()
label_names = [MarketState(i).name for i in label_counts.index]

plt.figure(figsize=(10, 5))
plt.bar(label_names, label_counts.values)
plt.title('Market State Distribution')
plt.xlabel('Market State')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\nLabel distribution:")
for i, count in label_counts.items():
    print(f"  {MarketState(i).name}: {count} ({count/len(df)*100:.1f}%)")

In [ ]:
# Prepare data for training
data = prepare_data_for_training(
    df,
    lookback=lookback,
    train_ratio=0.7,
    val_ratio=0.15
)

print(f"Training samples: {len(data['X_train'])}")
print(f"Validation samples: {len(data['X_val'])}")
print(f"Test samples: {len(data['X_test'])}")
print(f"Feature shape: {data['X_train'].shape}")

## 3. Model Training

In [ ]:
# Initialize model
model = ProtoPNet(
    input_channels=data['X_train'].shape[1],  # Number of features
    sequence_length=lookback,
    latent_dim=128,
    num_classes=5,
    prototypes_per_class=2,
)

print(f"Model architecture:")
print(f"  Input: {model.input_channels} features x {model.sequence_length} timesteps")
print(f"  Latent dim: {model.latent_dim}")
print(f"  Classes: {model.num_classes}")
print(f"  Prototypes: {model.num_prototypes} ({model.prototypes_per_class} per class)")

In [ ]:
# Initialize trainer
trainer = ProtoPNetTrainer(
    model=model,
    device=device,
    learning_rate=1e-3,
    lambda_clst=0.8,
    lambda_sep=-0.08,
)

# Train model
history = trainer.train(
    X_train=data['X_train'],
    y_train=data['y_train'],
    X_val=data['X_val'],
    y_val=data['y_val'],
    epochs=50,
    batch_size=32,
    push_epochs=[20, 40],
    early_stopping_patience=15,
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'], label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 4. Model Evaluation

In [ ]:
# Evaluate on test set
eval_results = trainer.evaluate(
    X_test=data['X_test'],
    y_test=data['y_test']
)

print(f"Test Accuracy: {eval_results['accuracy']:.4f}")
print(f"Mean Confidence: {eval_results['mean_confidence']:.4f}")
print("\nPer-class metrics:")
for class_name, metrics in eval_results['class_metrics'].items():
    print(f"  {class_name}: Acc={metrics['accuracy']:.4f}, Support={metrics['support']}")

In [ ]:
# Plot confusion matrix
confusion = np.array(eval_results['confusion_matrix'])
class_names = [MarketState(i).name for i in range(5)]

plt.figure(figsize=(10, 8))
plt.imshow(confusion, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()

tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)

# Add text annotations
thresh = confusion.max() / 2.
for i in range(confusion.shape[0]):
    for j in range(confusion.shape[1]):
        plt.text(j, i, format(confusion[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if confusion[i, j] > thresh else "black")

plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 5. Prototype Analysis

In [ ]:
# Analyze prototypes
proto_analysis = trainer.get_prototype_analysis(
    X=data['X_test'],
    y=data['y_test']
)

print("Prototype Analysis:")
for proto in proto_analysis['prototype_analysis']:
    print(f"\nPrototype {proto['prototype_idx']} (Class: {proto['class']})")
    print(f"  Mean similarity: {proto['mean_similarity']:.4f}")
    print(f"  Max similarity: {proto['max_similarity']:.4f}")
    print("  Activations by class:")
    for class_name, activation in proto['class_activations'].items():
        print(f"    {class_name}: {activation:.4f}")

In [ ]:
# Visualize prototype similarity patterns
sim_matrix = proto_analysis['similarity_matrix']

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for proto_idx in range(min(10, sim_matrix.shape[1])):
    ax = axes[proto_idx]
    proto_class = MarketState(proto_idx // 2).name
    
    # Plot similarity distribution by true class
    for class_idx in range(5):
        mask = data['y_test'] == class_idx
        ax.hist(sim_matrix[mask, proto_idx], alpha=0.5, 
                label=MarketState(class_idx).name, bins=30)
    
    ax.set_title(f'Prototype {proto_idx}\n({proto_class})')
    ax.set_xlabel('Similarity')
    ax.set_ylabel('Count')
    if proto_idx == 0:
        ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. Backtesting

In [ ]:
# Initialize backtester
backtester = PrototypeBacktester(
    model=model,
    device=device,
    confidence_threshold=0.6,
    position_sizing='confidence',
    initial_capital=100000,
    commission_pct=0.001,
    slippage_pct=0.0005,
)

# Generate signals on test data
test_start_idx = len(data['X_train']) + len(data['X_val'])
test_timestamps = pd.to_datetime(df['timestamp'].iloc[test_start_idx + lookback:test_start_idx + lookback + len(data['X_test'])].values)
test_prices = df['close'].iloc[test_start_idx + lookback:test_start_idx + lookback + len(data['X_test'])].values

signals = backtester.generate_signals(
    X=data['X_test'],
    timestamps=test_timestamps,
    prices=test_prices
)

print("Signal distribution:")
print(signals['position'].value_counts())

In [ ]:
# Run backtest
results = backtester.run_backtest(
    signals=signals,
    holding_period=5
)

# Print report
print_backtest_report(results)

In [ ]:
# Plot equity curve
equity = np.array(results['equity_curve'])

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Equity curve
axes[0].plot(equity, label='Strategy')
axes[0].axhline(y=100000, color='gray', linestyle='--', label='Initial Capital')
axes[0].set_xlabel('Time Step')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].set_title('Equity Curve')
axes[0].legend()
axes[0].grid(True)

# Drawdown
peak = np.maximum.accumulate(equity)
drawdown = (peak - equity) / peak * 100

axes[1].fill_between(range(len(drawdown)), 0, drawdown, color='red', alpha=0.3)
axes[1].set_xlabel('Time Step')
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_title('Drawdown')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Analyze trade distribution
if results['trades']:
    trades_df = pd.DataFrame([
        {
            'entry_time': t.entry_time,
            'exit_time': t.exit_time,
            'direction': 'Long' if t.direction == 1 else 'Short',
            'pnl': t.pnl,
            'return_pct': t.return_pct * 100,
            'confidence': t.confidence,
            'predicted_state': MarketState(t.predicted_state).name,
        }
        for t in results['trades']
    ])
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # P&L distribution
    axes[0].hist(trades_df['return_pct'], bins=30, edgecolor='black')
    axes[0].axvline(x=0, color='red', linestyle='--')
    axes[0].set_xlabel('Return (%)')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Trade Return Distribution')
    
    # Confidence vs Return
    colors = ['green' if r > 0 else 'red' for r in trades_df['return_pct']]
    axes[1].scatter(trades_df['confidence'], trades_df['return_pct'], c=colors, alpha=0.5)
    axes[1].axhline(y=0, color='gray', linestyle='--')
    axes[1].set_xlabel('Confidence')
    axes[1].set_ylabel('Return (%)')
    axes[1].set_title('Confidence vs Return')
    
    # Cumulative P&L
    cumulative_pnl = trades_df['pnl'].cumsum()
    axes[2].plot(cumulative_pnl)
    axes[2].set_xlabel('Trade Number')
    axes[2].set_ylabel('Cumulative P&L ($)')
    axes[2].set_title('Cumulative P&L')
    axes[2].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    print("\nSample trades:")
    display(trades_df.head(10))

## 7. Interpretation: Why Did the Model Predict This?

One of the key advantages of prototype learning is interpretability. Let's examine a specific prediction.

In [ ]:
# Take a sample from test set
sample_idx = 100  # Change this to explore different samples
sample_x = torch.tensor(data['X_test'][sample_idx:sample_idx+1], dtype=torch.float32).to(device)
sample_y = data['y_test'][sample_idx]

# Get model prediction
model.eval()
with torch.no_grad():
    logits, similarities, distances = model(sample_x)
    probs = torch.softmax(logits, dim=1)
    pred = probs.argmax().item()
    confidence = probs.max().item()

similarities = similarities.cpu().numpy()[0]

print(f"Sample {sample_idx}:")
print(f"  True label: {MarketState(sample_y).name}")
print(f"  Predicted: {MarketState(pred).name}")
print(f"  Confidence: {confidence:.4f}")
print(f"\nPrototype similarities:")

# Show which prototypes activated
for proto_idx in range(len(similarities)):
    proto_class = MarketState(proto_idx // 2).name
    print(f"  Prototype {proto_idx} ({proto_class}): {similarities[proto_idx]:.4f}")

# Plot similarities
plt.figure(figsize=(10, 4))
colors = ['green' if i // 2 == pred else 'blue' if i // 2 == sample_y else 'gray' 
          for i in range(len(similarities))]
plt.bar(range(len(similarities)), similarities, color=colors)
plt.xlabel('Prototype Index')
plt.ylabel('Similarity')
plt.title(f'Prototype Similarities\n(True: {MarketState(sample_y).name}, Pred: {MarketState(pred).name})')
plt.xticks(range(len(similarities)), 
           [f'{i}\n({MarketState(i//2).name[:3]})' for i in range(len(similarities))],
           fontsize=8)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

1. **Data Loading**: Fetching cryptocurrency data from Bybit and stock data from Yahoo Finance
2. **Feature Engineering**: Computing technical indicators for prototype learning
3. **Model Training**: Training ProtoPNet with combined loss function
4. **Prototype Analysis**: Understanding which patterns the model learned
5. **Backtesting**: Evaluating the trading strategy based on prototype predictions
6. **Interpretation**: Explaining individual predictions using prototype similarities

Key advantages of prototype learning for trading:
- **Interpretability**: Understand why the model makes each prediction
- **Transparency**: Inspect learned patterns (prototypes)
- **Confidence**: Know when the model is uncertain
- **Adaptability**: Prototypes can be updated with new market patterns